### 1. Installation et Imports

In [14]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.evaluation import evaluate_policy
import os

In [15]:
# Créer les répertoires nécessaires pour les logs et vidéos
os.makedirs("../data/logs", exist_ok=True)
os.makedirs("../data/videos", exist_ok=True)

print("✓ Répertoires créés")

✓ Répertoires créés


### 2. Exploration de l'Environnement

In [16]:
# Créer l'environnement LunarLander-v3
env = gym.make("LunarLander-v3")

print("┌─ EXPLORATION DE L'ENVIRONNEMENT LunarLander-v3 ─┐")
print(f"\nEspace d'OBSERVATION:")
print(f"  Type: {type(env.observation_space)}")
print(f"  Forme: {env.observation_space.shape}")
print(f"  Min: {env.observation_space.low}")
print(f"  Max: {env.observation_space.high}")

print(f"\nEspace d'ACTIONS:")
print(f"  Type: {type(env.action_space)}")
print(f"  Nombre d'actions: {env.action_space.n}")
print(f"  Actions disponibles: {list(range(env.action_space.n))}")
print(f"    - 0: Ne rien faire")
print(f"    - 1: Augmenter le moteur gauche")
print(f"    - 2: Augmenter le moteur droit")
print(f"    - 3: Augmenter les deux moteurs")
print("└" + "─" * 47 + "┘")

┌─ EXPLORATION DE L'ENVIRONNEMENT LunarLander-v3 ─┐

Espace d'OBSERVATION:
  Type: <class 'gymnasium.spaces.box.Box'>
  Forme: (8,)
  Min: [ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ]
  Max: [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ]

Espace d'ACTIONS:
  Type: <class 'gymnasium.spaces.discrete.Discrete'>
  Nombre d'actions: 4
  Actions disponibles: [0, 1, 2, 3]
    - 0: Ne rien faire
    - 1: Augmenter le moteur gauche
    - 2: Augmenter le moteur droit
    - 3: Augmenter les deux moteurs
└───────────────────────────────────────────────┘


**Conclusion:** L'espace d'action est **discret** avec 4 actions possibles. Selon les recommandations, **DQN est l'algorithme adapté** pour les espaces d'action discrets.

### 3. Définition de la Baseline (Agent Aléatoire)

In [19]:
# Mesure de performance d'un agent aléatoire sur 50 épisodes
print("\n📊 ÉVALUATION DE LA BASELINE (Agent Aléatoire)\n")

# Agent aléatoire: prendre des actions au hasard
def random_agent(env, num_episodes=50, seed=42):
    """Teste les performances d'un agent qui choisit des actions aléatoires."""
    episodes_rewards = []
    
    for episode in range(1, num_episodes + 1):
        obs, info = env.reset(seed=seed + episode)  # Graine déterministe par épisode
        done = False
        episode_reward = 0
        
        while not done:
            action = env.action_space.sample()  # Action aléatoire
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            done = terminated or truncated
        
        episodes_rewards.append(episode_reward)
        if episode % 10 == 0:
            print(f"  Épisode {episode:2d}/{num_episodes} | Reward: {episode_reward:7.2f}")
    
    return episodes_rewards

baseline_rewards = random_agent(env, num_episodes=50)
baseline_mean = np.mean(baseline_rewards)
baseline_std = np.std(baseline_rewards)

print(f"\n{'─' * 50}")
print(f"RÉSUMÉ - BASELINE (Agent Aléatoire):")
print(f"  Reward moyenne: {baseline_mean:.2f} ± {baseline_std:.2f}")
print(f"  Min: {np.min(baseline_rewards):.2f} | Max: {np.max(baseline_rewards):.2f}")
print(f"{'─' * 50}\n")


📊 ÉVALUATION DE LA BASELINE (Agent Aléatoire)

  Épisode 10/50 | Reward:  -97.32
  Épisode 20/50 | Reward:  -83.10
  Épisode 30/50 | Reward: -147.26
  Épisode 40/50 | Reward: -271.35
  Épisode 50/50 | Reward: -212.96

──────────────────────────────────────────────────
RÉSUMÉ - BASELINE (Agent Aléatoire):
  Reward moyenne: -193.69 ± 109.46
  Min: -435.18 | Max: 15.87
──────────────────────────────────────────────────



### 4. Entraînement DQN avec Paramètres Par Défaut

Nous entraînons un agent DQN avec les paramètres par défaut de Stable-Baselines3 pendant 100,000 timesteps.

In [20]:
print("🚀 ENTRAÎNEMENT DQN (100,000 timesteps)\n")

# Créer un nouvel environnement frais pour l'entraînement
env_dqn = gym.make("LunarLander-v3")
env_dqn.reset(seed=42)  # ← Fixer graine environnement

# Initialiser le modèle DQN avec les PARAMÈTRES PAR DÉFAUT de Stable-Baselines3
# Les paramètres par défaut sont optimisés pour LunarLander
model_dqn = DQN(
    "MlpPolicy",           # Réseau de neurones Multi-Layer Perceptron
    env_dqn,
    seed=42,               # ← Fixer graine modèle
    verbose=1,             # Affiche les logs d'entraînement
    tensorboard_log="../data/logs/"  # Sauvegarde des logs pour TensorBoard
)

# Entraînement du modèle
model_dqn.learn(total_timesteps=100_000)

print("\n✓ Entraînement DQN terminé!")

🚀 ENTRAÎNEMENT DQN (100,000 timesteps)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ../data/logs/DQN_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 91       |
|    ep_rew_mean      | -208     |
|    exploration_rate | 0.965    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 2793     |
|    time_elapsed     | 0        |
|    total_timesteps  | 364      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 2.14     |
|    n_updates        | 65       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 88.9     |
|    ep_rew_mean      | -156     |
|    exploration_rate | 0.932    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 3802     |
|    time_elapsed     | 0      

### 5. Évaluation DQN et Comparaison avec la Baseline

In [21]:
# Évaluation du modèle DQN entraîné sur 50 épisodes
print("\n📊 ÉVALUATION DU MODÈLE DQN (50 épisodes)\n")

mean_reward_dqn, std_reward_dqn = evaluate_policy(
    model_dqn, 
    env_dqn, 
    n_eval_episodes=50
)

print(f"\nRésultat DQN:")
print(f"  Reward moyenne: {mean_reward_dqn:.2f} ± {std_reward_dqn:.2f}")


📊 ÉVALUATION DU MODÈLE DQN (50 épisodes)


Résultat DQN:
  Reward moyenne: 18.13 ± 166.92


**Comparaison avec la baseline aléatoire:**

In [9]:
# Comparaison complète: Baseline aléatoire vs DQN entraîné
print("\n" + "═" * 60)
print("COMPARAISON FINALE: BASELINE vs DQN")
print("═" * 60)

print(f"\n📍 Agent Aléatoire (Baseline):")
print(f"   Reward moyenne: {baseline_mean:7.2f} ± {baseline_std:.2f}")

print(f"\n🎯 Agent DQN (Modèle Entraîné):")
print(f"   Reward moyenne: {mean_reward_dqn:7.2f} ± {std_reward_dqn:.2f}")

improvement = mean_reward_dqn - baseline_mean
improvement_percent = (improvement / abs(baseline_mean)) * 100 if baseline_mean != 0 else 0

print(f"\n📈 AMÉLIORATION DQN vs Baseline:")
print(f"   Différence: {improvement:+.2f} points ({improvement_percent:+.1f}%)")

if mean_reward_dqn > baseline_mean:
    print(f"   ✓ DQN est meilleur! ({improvement:.2f} points de plus)")
else:
    print(f"   ✗ DQN est moins bon (à investiguer)")

print("═" * 60 + "\n")


════════════════════════════════════════════════════════════
COMPARAISON FINALE: BASELINE vs DQN
════════════════════════════════════════════════════════════

📍 Agent Aléatoire (Baseline):
   Reward moyenne: -181.52 ± 100.32

🎯 Agent DQN (Modèle Entraîné):
   Reward moyenne:   18.13 ± 166.92

📈 AMÉLIORATION DQN vs Baseline:
   Différence: +199.65 points (+110.0%)
   ✓ DQN est meilleur! (199.65 points de plus)
════════════════════════════════════════════════════════════



### 6. Entraînement PPO avec Paramètres Par Défaut

Nous entraînons un agent PPO avec les paramètres par défaut de Stable-Baselines3 pendant 100,000 timesteps, pour comparer équitablement avec DQN.

In [22]:
# Entraînement PPO avec paramètres par défaut
print("🚀 ENTRAÎNEMENT PPO (100,000 timesteps)\n")

# Créer un nouvel environnement frais pour l'entraînement
env_ppo = gym.make("LunarLander-v3")
env_ppo.reset(seed=42)  # ← Fixer graine environnement

# Initialiser le modèle PPO avec les PARAMÈTRES PAR DÉFAUT de Stable-Baselines3
model_ppo = PPO(
    "MlpPolicy",           # Réseau de neurones Multi-Layer Perceptron
    env_ppo,
    verbose=1,             # Affiche les logs d'entraînement
    tensorboard_log="../data/logs/"  # Sauvegarde des logs pour TensorBoard
)

model_ppo.learn(total_timesteps=100_000)
print("\n✓ Entraînement PPO terminé!")

# Évaluation du modèle PPO
print("\n📊 ÉVALUATION DU MODÈLE PPO (50 épisodes)\n")

mean_reward_ppo, std_reward_ppo = evaluate_policy(
    model_ppo,
    env_ppo,
    n_eval_episodes=50
)

print(f"\nRésultat PPO:")
print(f"  Reward moyenne: {mean_reward_ppo:.2f} ± {std_reward_ppo:.2f}")

🚀 ENTRAÎNEMENT PPO (100,000 timesteps)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ../data/logs/PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96.1     |
|    ep_rew_mean     | -180     |
| time/              |          |
|    fps             | 6839     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 96.1        |
|    ep_rew_mean          | -176        |
| time/                   |             |
|    fps                  | 4846        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.014306281 |
|    clip_fraction        | 0.0646

### 7. Comparaison: DQN vs PPO

Comparison des performances des deux algorithmes sur l'espace discret de LunarLander-v3.

In [23]:
# Comparaison DQN vs PPO
print("\n" + "═" * 60)
print("COMPARAISON: DQN vs PPO")
print("═" * 60)

print(f"\n🎯 DQN (Value-Based):")
print(f"   Reward moyenne: {mean_reward_dqn:7.2f} ± {std_reward_dqn:.2f}")

print(f"\n🔵 PPO (Policy-Based):")
print(f"   Reward moyenne: {mean_reward_ppo:7.2f} ± {std_reward_ppo:.2f}")

diff = mean_reward_ppo - mean_reward_dqn
if mean_reward_ppo > mean_reward_dqn:
    print(f"\n   ✓ PPO est meilleur de {diff:+.2f} points")
    print(f"   📊 Analyse: Bien que PPO soit adapté aux espaces continus,")
    print(f"      sa robustesse et sa stabilité lui permettent de surpasser DQN")
    print(f"      avec les hyperparamètres par défaut sur cet espace discret.")
else:
    diff = mean_reward_dqn - mean_reward_ppo
    print(f"\n   ✓ DQN est meilleur de {diff:+.2f} points")
    print(f"   📊 Analyse: DQN est plus adapté aux espaces discrets")
    print(f"      et montre une meilleure performance que PPO.")

print("═" * 60 + "\n")


════════════════════════════════════════════════════════════
COMPARAISON: DQN vs PPO
════════════════════════════════════════════════════════════

🎯 DQN (Value-Based):
   Reward moyenne:   18.13 ± 166.92

🔵 PPO (Policy-Based):
   Reward moyenne:  -88.29 ± 31.60

   ✓ DQN est meilleur de +106.43 points
   📊 Analyse: DQN est plus adapté aux espaces discrets
      et montre une meilleure performance que PPO.
════════════════════════════════════════════════════════════



In [12]:
import json
from pathlib import Path

# Sauvegarder les résultats du notebook 1 pour utilisation dans notebook 2
baseline_results = {
    "baseline_mean": float(baseline_mean),
    "baseline_std": float(baseline_std),
    "dqn_mean": float(mean_reward_dqn),
    "dqn_std": float(std_reward_dqn),
    "ppo_mean": float(mean_reward_ppo),
    "ppo_std": float(std_reward_ppo)
}

# Sauvegarder dans data/
results_file = Path("../data/baseline_results.json")
results_file.parent.mkdir(parents=True, exist_ok=True)

with open(results_file, "w") as f:
    json.dump(baseline_results, f, indent=2)

print("✓ Résultats baseline sauvegardés dans data/baseline_results.json")
print("\nRésumé exporté:")
for key, value in baseline_results.items():
    print(f"  {key}: {value:.2f}")

✓ Résultats baseline sauvegardés dans data/baseline_results.json

Résumé exporté:
  baseline_mean: -181.52
  baseline_std: 100.32
  dqn_mean: 18.13
  dqn_std: 166.92
  ppo_mean: -88.29
  ppo_std: 31.60
